# Beras Weekly Price Forecasting — XGBoost Walk-Forward Training

Standalone training notebook for **Beras Medium** and **Beras Premium** weekly prices across all Maluku regencies.

**Feature set mirrors the production trainer** (`sistem/services/trainer.py`):
- Self price features: lags, log-return, change_pct lags, LOCF streak
- Seasonality: OHE week-number + sin/cos circular encoding
- Cross-commodity (grains group): the other beras variety's change_pct
- Cross-city hub reference (kab 8101 = Maluku Tengah): contemporaneous change_pct, spatial lags, rolling volatility, hub-to-local price spread

**No Django ORM required** — reads from exported CSVs in `notebooks/data/`.

To switch the target commodity, change `TARGET_COMMODITY` in the config cell below.

In [ ]:
# ── CONFIG ────────────────────────────────────────────────────────────────────
TARGET_COMMODITY = "Beras Medium"   # or "Beras Premium"
REFERENCE_KAB    = "8101"           # hub city (Maluku Tengah) — production default
HORIZON          = 4                # train H+1 … H+H models
N_SPLITS         = 5                # walk-forward CV folds
MIN_ROWS         = 26               # skip kabupaten with fewer observations
DATA_DIR         = "data"           # relative to this notebook's directory
# ──────────────────────────────────────────────────────────────────────────────

In [ ]:
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from sklearn.model_selection import TimeSeriesSplit
from xgboost import XGBRegressor

warnings.filterwarnings("ignore", category=UserWarning)
pd.set_option("display.float_format", "{:.3f}".format)
plt.rcParams["figure.dpi"] = 110

DATA_PATH = Path(DATA_DIR)
OTHER_COMMODITY = "Beras Premium" if TARGET_COMMODITY == "Beras Medium" else "Beras Medium"
TARGET_SLUG = TARGET_COMMODITY.lower().replace(" ", "_")
OTHER_SLUG  = OTHER_COMMODITY.lower().replace(" ", "_")

print(f"Target : {TARGET_COMMODITY}")
print(f"Other  : {OTHER_COMMODITY}")
print(f"Ref hub: kabupaten {REFERENCE_KAB}")

## 1 · Load Data

In [ ]:
def load_csv(slug: str) -> pd.DataFrame:
    path = DATA_PATH / f"{slug}_weekly.csv"
    df = pd.read_csv(path, dtype={"kabupaten_kode": str})
    df["periode_start"] = pd.to_datetime(df["periode_start"])
    df["periode_end"]   = pd.to_datetime(df["periode_end"])
    return df

df_target = load_csv(TARGET_SLUG)
df_other  = load_csv(OTHER_SLUG)

for name, df in [(TARGET_COMMODITY, df_target), (OTHER_COMMODITY, df_other)]:
    kabs = df["kabupaten_kode"].nunique()
    date_range = f"{df['periode_start'].min().date()} → {df['periode_end'].max().date()}"
    locf_pct = df["is_locf"].mean() * 100
    print(f"\n{name}: {len(df):,} rows | {kabs} kabupaten | {date_range} | LOCF {locf_pct:.1f}%")
    print(df[["harga_lkv","change_pct","harga_lag1","harga_lag2","harga_lag3"]].isnull().mean().rename("null_rate"))

# Sanity check: reference kabupaten must be in both files
for name, df in [(TARGET_COMMODITY, df_target), (OTHER_COMMODITY, df_other)]:
    assert REFERENCE_KAB in df["kabupaten_kode"].values, \
        f"Reference kabupaten {REFERENCE_KAB} missing from {name} CSV!"
print(f"\nReference kabupaten {REFERENCE_KAB} present in both files.")

## 2 · EDA

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(13, 7), sharex=True)

for ax, (name, df) in zip(axes, [(TARGET_COMMODITY, df_target), (OTHER_COMMODITY, df_other)]):
    weekly = df.groupby("periode_start")["harga_lkv"].agg(["median", "min", "max"])
    ax.fill_between(weekly.index, weekly["min"], weekly["max"], alpha=0.2, label="min–max range")
    ax.plot(weekly.index, weekly["median"], lw=1.8, label="median across kabupaten")
    ax.set_title(f"{name} — weekly LKV price (IDR/kg)")
    ax.set_ylabel("IDR")
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# LOCF heatmap: how often is each kabupaten carrying forward a stale price?
locf_pivot = (
    df_target
    .assign(year=df_target["periode_start"].dt.year)
    .groupby(["kabupaten_nama", "year"])["is_locf"]
    .mean()
    .unstack("year")
)

fig, ax = plt.subplots(figsize=(10, max(4, len(locf_pivot) * 0.4)))
sns.heatmap(locf_pivot, annot=True, fmt=".0%", cmap="YlOrRd", linewidths=0.4,
            cbar_kws={"label": "LOCF rate"}, ax=ax)
ax.set_title(f"{TARGET_COMMODITY} — LOCF rate per kabupaten / year")
ax.set_ylabel("")
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, (name, df) in zip(axes, [(TARGET_COMMODITY, df_target), (OTHER_COMMODITY, df_other)]):
    chg = df["change_pct"].dropna()
    ax.hist(chg.clip(-10, 10), bins=60, edgecolor="none", alpha=0.8)
    ax.axvline(0, color="k", lw=0.8, ls="--")
    ax.set_title(f"{name}\nchange_pct distribution (clipped ±10%)")
    ax.set_xlabel("%")
    ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 3 · Feature Engineering — `build_stream_df`

Mirrors `build_training_dataframe()` (aggregator.py:619) + `_build_features()` (trainer.py:72) in pure pandas.

In [ ]:
def build_stream_df(kab_kode: str) -> pd.DataFrame:
    """
    Assemble a feature-rich training DataFrame for one kabupaten.
    Returns a DataFrame sorted by period, ready for (X, y) extraction.
    """
    PERIOD_COLS = ["periode_tahun", "periode_nomor", "periode_start", "periode_end"]
    MERGE_ON    = ["periode_tahun", "periode_nomor"]

    # ── Step 1: local self rows ────────────────────────────────────────────────
    local = (
        df_target[df_target["kabupaten_kode"] == kab_kode]
        .sort_values(MERGE_ON)
        .reset_index(drop=True)
    )
    if local.empty:
        return pd.DataFrame()

    price_cols = ["harga_lkv", "is_locf", "is_up", "harga_delta",
                  "change_pct", "harga_lag1", "harga_lag2", "harga_lag3"]
    df = local[PERIOD_COLS + price_cols].copy()
    df = df.rename(columns={c: f"self_{c}" for c in price_cols})

    # ── Step 2: lagged self change_pct (aggregator.py:654-658) ────────────────
    for lag in (1, 2):
        df[f"self_change_pct_lag{lag}"] = df["self_change_pct"].shift(lag)

    # ── Step 3: LOCF streak (aggregator.py:688-694) ────────────────────────────
    streak, count = [], 0
    for v in df["self_is_locf"]:
        count = count + 1 if v else 0
        streak.append(count)
    df["self_locf_streak"] = streak

    # ── Step 4: self log-return (trainer.py:96-99) ────────────────────────────
    cur  = df["self_harga_lkv"].replace(0, np.nan)
    prev = df["self_harga_lag1"].replace(0, np.nan)
    df["self_log_return"] = np.log(cur / prev).fillna(0.0)

    # ── Step 5: seasonality (aggregator.py:737, trainer.py:120-125) ───────────
    nomor = df["periode_nomor"].astype(int)
    ohe = pd.get_dummies(nomor, prefix="period_num", dtype=int)
    df  = pd.concat([df, ohe], axis=1)
    df["sin_week"] = np.sin(2 * np.pi * nomor / 52)
    df["cos_week"] = np.cos(2 * np.pi * nomor / 52)

    # ── Step 6: cross-commodity — grains group (trainer.py:132-157) ──────────
    other_local = (
        df_other[df_other["kabupaten_kode"] == kab_kode]
        .sort_values(MERGE_ON)
        [[*MERGE_ON, "change_pct"]]
        .rename(columns={"change_pct": "other_change_pct"})
    )
    if not other_local.empty:
        df = df.merge(other_local, on=MERGE_ON, how="left")
        # grains group mean = mean of target + other (both varieties)
        df["group_grains_mean_change_pct"] = df[["self_change_pct", "other_change_pct"]].mean(axis=1)
        # inflationary pressure: fraction of grains currently rising
        df["inflationary_pressure"] = (
            (df[["self_change_pct", "other_change_pct"]] > 0).mean(axis=1)
        )
        df["other_change_pct"] = df["other_change_pct"].fillna(0.0)
        df["group_grains_mean_change_pct"] = df["group_grains_mean_change_pct"].fillna(0.0)
        df["inflationary_pressure"]        = df["inflationary_pressure"].fillna(0.0)

    # ── Step 7: cross-city reference hub (aggregator.py:697-729, trainer.py:159-171)
    if kab_kode != REFERENCE_KAB:
        ref = (
            df_target[df_target["kabupaten_kode"] == REFERENCE_KAB]
            .sort_values(MERGE_ON)
            [[*MERGE_ON, "harga_lkv", "change_pct"]]
            .rename(columns={
                "harga_lkv":  f"harga_lkv_ref_{REFERENCE_KAB}",
                "change_pct": f"change_pct_ref_{REFERENCE_KAB}",
            })
        )
        df = df.merge(ref, on=MERGE_ON, how="left")

        ref_chg = f"change_pct_ref_{REFERENCE_KAB}"
        ref_lkv = f"harga_lkv_ref_{REFERENCE_KAB}"

        # Spatial lags — shipping delay 7–14 days (aggregator.py:718-724)
        for lag in (1, 2):
            df[f"{ref_chg}_lag{lag}"] = df[ref_chg].shift(lag)

        # Rolling volatility — logistics disruption proxy (aggregator.py:728-729)
        df[f"{ref_chg}_vol3"] = df[ref_chg].rolling(3, min_periods=1).std().fillna(0.0)

        # Hub-to-local price spread — catch-up pressure (trainer.py:164-171)
        df[f"hub_price_gap_{REFERENCE_KAB}"] = df[ref_lkv] - df["self_harga_lkv"]

        # Fill cross-city NaNs with 0 so local-data rows are not dropped
        cross_cols = [c for c in df.columns if f"_ref_{REFERENCE_KAB}" in c
                      or c.startswith("hub_price_gap_")]
        df[cross_cols] = df[cross_cols].fillna(0.0)

    return df.reset_index(drop=True)


# Quick sanity check on one kabupaten
_sample = build_stream_df("8102")
print(f"Sample stream (8102): {len(_sample)} rows, {_sample.shape[1]} columns")
print(_sample.columns.tolist())

## 4 · Walk-Forward Training

One `XGBRegressor` per (kabupaten, horizon). Hyperparameters identical to `train_stream()` in `trainer.py:315–334`.  
Target: **cumulative log-return** `ln(P_{t+h} / P_t)` — same formulation as `trainer.py:190–196`.

In [ ]:
SKIP_COLS = ["periode_tahun", "periode_nomor", "periode_start", "periode_end"]
XGB_PARAMS = dict(
    n_estimators=100, max_depth=4, learning_rate=0.1,
    subsample=0.8, colsample_bytree=0.8, random_state=42, verbosity=0,
)

all_kabupaten = sorted(df_target["kabupaten_kode"].unique())
results   = {}   # {kab_kode: {h: {"mape_price": ..., "mae": ...}}}
h1_models = {}   # store H+1 model per kabupaten for feature importance
kab_names = dict(zip(df_target["kabupaten_kode"], df_target["kabupaten_nama"]))

print(f"Training {len(all_kabupaten)} kabupaten × H+1…H+{HORIZON} models\n")

for kab_kode in all_kabupaten:
    stream = build_stream_df(kab_kode)
    if stream.empty:
        print(f"  SKIP {kab_kode}: no data")
        continue

    feat_cols = [c for c in stream.columns if c not in SKIP_COLS + ["kabupaten_kode", "kabupaten_nama"]]
    X_full = stream[feat_cols].copy()
    harga  = stream["self_harga_lkv"]

    kab_metrics = {}
    for h in range(1, HORIZON + 1):
        future_p  = harga.shift(-h)
        current_p = harga.replace(0, np.nan)
        y_full    = np.log(future_p / current_p)

        valid = X_full.notna().all(axis=1) & y_full.notna()
        X, y  = X_full[valid], y_full[valid]

        if len(X) < MIN_ROWS:
            continue

        test_size = max(5, len(X) // 10)
        tscv = TimeSeriesSplit(n_splits=N_SPLITS, test_size=test_size)

        oof_true, oof_pred, oof_idx = [], [], []
        for train_idx, val_idx in tscv.split(X):
            m = XGBRegressor(**XGB_PARAMS)
            m.fit(X.iloc[train_idx], y.iloc[train_idx])
            oof_true.extend(y.iloc[val_idx].tolist())
            oof_pred.extend(m.predict(X.iloc[val_idx]).tolist())
            oof_idx.extend(X.iloc[val_idx].index.tolist())

        # Final model on all data
        final_model = XGBRegressor(**XGB_PARAMS)
        final_model.fit(X, y)
        if h == 1:
            h1_models[kab_kode] = (final_model, list(X.columns))

        # MAPE on reconstructed price level (trainer.py:340-354)
        if oof_idx:
            idx_s  = pd.Index(oof_idx)
            cur_p  = harga.loc[idx_s].values
            act_p  = harga.shift(-h).loc[idx_s].values
            pred_p = cur_p * np.exp(np.array(oof_pred))
            mask   = ~np.isnan(act_p) & (act_p > 0)
            mape   = float(np.mean(np.abs(act_p[mask] - pred_p[mask]) / act_p[mask]) * 100) if mask.any() else 0.0
        else:
            mape = 0.0

        mae = float(np.mean(np.abs(np.array(oof_true) - np.array(oof_pred))))
        kab_metrics[h] = {"mape_price": mape, "mae": mae}

    if kab_metrics:
        results[kab_kode] = kab_metrics
        horizons_str = "  ".join(f"H+{h}: {v['mape_price']:.2f}%" for h, v in kab_metrics.items())
        print(f"  OK  {kab_kode} {kab_names.get(kab_kode, '')[:28]:<28}  {horizons_str}")
    else:
        print(f"  SKIP {kab_kode}: too few rows")

print(f"\nDone. {len(results)}/{len(all_kabupaten)} kabupaten trained.")

## 5 · Results Table

In [ ]:
rows = []
for kab_kode, metrics in results.items():
    row = {"kabupaten_kode": kab_kode, "kabupaten_nama": kab_names.get(kab_kode, kab_kode)}
    for h in range(1, HORIZON + 1):
        row[f"MAPE H+{h} (%)"] = metrics.get(h, {}).get("mape_price", np.nan)
    rows.append(row)

results_df = pd.DataFrame(rows).set_index("kabupaten_nama").drop(columns="kabupaten_kode")
print(f"\n{TARGET_COMMODITY} — MAPE by kabupaten and horizon\n")
print(results_df.to_string())

print("\n── Summary statistics ──")
print(results_df.describe().loc[["mean", "min", "max"]])

In [ ]:
# Best and worst regencies at H+1
h1_col = "MAPE H+1 (%)"
ranked = results_df[h1_col].sort_values()

print(f"Best 3 kabupaten (lowest MAPE at H+1):")
print(ranked.head(3).to_string())
print(f"\nWorst 3 kabupaten (highest MAPE at H+1):")
print(ranked.tail(3).to_string())

# Bar chart
fig, ax = plt.subplots(figsize=(12, max(4, len(results_df) * 0.35)))
mape_cols = [f"MAPE H+{h} (%)" for h in range(1, HORIZON + 1)]
results_df[mape_cols].sort_values(h1_col).plot.barh(ax=ax, width=0.75)
ax.set_title(f"{TARGET_COMMODITY} — MAPE per kabupaten and horizon")
ax.set_xlabel("MAPE (%)")
ax.grid(True, axis="x", alpha=0.4)
plt.tight_layout()
plt.show()

## 6 · Feature Importance (H+1 model)

Using a representative non-hub kabupaten so cross-city features are present.

In [ ]:
# Pick first trained kabupaten that is not the reference hub
ref_kab = next((k for k in results if k != REFERENCE_KAB and k in h1_models), None)
if ref_kab is None:
    ref_kab = next(iter(h1_models), None)

if ref_kab is not None:
    model, feature_names = h1_models[ref_kab]
    importance = pd.Series(model.feature_importances_, index=feature_names)
    top15 = importance.sort_values(ascending=False).head(15)

    fig, ax = plt.subplots(figsize=(9, 5))
    top15.sort_values().plot.barh(ax=ax, color="steelblue")
    ax.set_title(
        f"{TARGET_COMMODITY} — H+1 feature importance\n"
        f"kabupaten {ref_kab} ({kab_names.get(ref_kab, '')})"
    )
    ax.set_xlabel("Importance")
    ax.grid(True, axis="x", alpha=0.4)
    plt.tight_layout()
    plt.show()

    print("\nTop 15 features:")
    print(top15.to_string())
else:
    print("No trained models available.")